# RegimeShift: Market-Regime Asset Allocation System
## IIT Bombay Summer Quant 2026 — Final Submission

This notebook reproduces the complete RegimeShift pipeline:
data → features → HMM → portfolio → backtest → metrics → charts.
All outputs are computed from the **real market dataset** loaded at
the top of this notebook.

**Dataset SHA-256:** `bab5a877c16b189640504d150d2c37b805c8f17a2426e588c9a71b03e15391b7`

### 1. Objective

Build a leakage-free market-regime detection system (Bull, Bear, Crisis) that dynamically rebalances a multi-asset portfolio (NIFTY 50 equity, GOLDBEES.NS gold, LIQUIDBEES.NS bond) while accounting for realistic transaction costs (5–10 bps).

**Active tickers (from config.py):**
- Equity: `^NSEI` (NIFTY 50)
- Gold: `GOLDBEES.NS` (Nippon India ETF Gold Bees)
- Bond: `LIQUIDBEES.NS` — LIQUIDBEES.NS — Nippon India Liquid Bees, INR-denominated liquid-bond / cash-equivalent proxy (NOT a sovereign bond or 10-year G-Sec; short duration, no long-term rate hedge)
- VIX: `omitted` (optional; omitted)

**Transaction cost (official):** 5.0 bps  | **Rebalance frequency:** 21 days  | **Train window:** 252 days

### 2. Reproducibility and Configuration

All random seeds are fixed; all HMM training is on rolling windows through `t-1`. VIX is optional and never allocated.

In [1]:
from regime_shift.config import RegimeShiftConfig
config = RegimeShiftConfig()
config.transaction_cost_bps = 5.0
config.rebalance_frequency = 21
config.train_window = 252

print("Equity:     ^NSEI")
print("Gold:       GOLDBEES.NS")
print("Bond:       LIQUIDBEES.NS")
print("VIX:        omitted")
print("Cost:       5.0 bps")
print("Train window: 252 days")
print("Rebalance freq: 21 days")
print("HMM:        3 states, covariance_type='diag'")
print("Risk-free rate: 0.0")
print("Annualisation: 252")


Equity:     ^NSEI
Gold:       GOLDBEES.NS
Bond:       LIQUIDBEES.NS
VIX:        omitted
Cost:       5.0 bps
Train window: 252 days
Rebalance freq: 21 days
HMM:        3 states, covariance_type='diag'
Risk-free rate: 0.0
Annualisation: 252


### 3. Real Asset Universe and Actual Tickers

**No synthetic prices are generated in this notebook.**
The same real dataset is used for the official CLI and cost-sensitivity runs.

| Property | Value |
|---|---|
| File | `data/submission_market_data.csv` |
| SHA-256 | `bab5a877c16b189640504d150d2c37b805c8f17a2426e588c9a71b03e15391b7` |
| First date | `2010-01-04` |
| Last date | `2026-07-27` |
| Row count | `4090` |
| Columns | `equity, gold, bond` |
| VIX included | `False` |
| Total forward-filled cells | `0` |
| Filled per asset | `{'equity': 0, 'gold': 0, 'bond': 0}` |
| Dates dropped (residual NaNs) | `0` |

**Bond proxy:** LIQUIDBEES.NS — Nippon India Liquid Bees, INR-denominated liquid-bond / cash-equivalent proxy (NOT a sovereign bond or 10-year G-Sec; short duration, no long-term rate hedge)

In [2]:
from regime_shift.data import load_market_data_csv
prices = load_market_data_csv(path="data/submission_market_data.csv", config=config)

print(f"Price data: {len(prices)} rows")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Columns: {list(prices.columns)}")
display(prices.head())
display(prices.describe())

print(f"Forward-filled cells: 0")
print("Per-asset fill counts:", {'equity': 0, 'gold': 0, 'bond': 0})
print(f"Dates dropped: 0")
print(f"Data file SHA-256: bab5a877c16b189640504d150d2c37b805c8f17a2426e588c9a71b03e15391b7")


Column 'bond' has 57% values equal to their successor - possible backward-fill detected. Verify raw data source.


Price data: 4090 rows
Date range: 2010-01-04 to 2026-07-27
Columns: ['equity', 'gold', 'bond']


,equity,gold,bond
Date,,,
2010-01-04,5232.200195,16.549900,589.801575
2010-01-05,5277.899902,16.680000,589.801575
2010-01-06,5281.799805,16.600000,589.801575
2010-01-07,5263.100098,16.572001,589.807129
2010-01-08,5244.750000,16.490000,589.801575


,equity,gold,bond
count,4090.000000,4090.000000,4090.000000
mean,12252.154996,39.347400,760.585919
std,6534.918209,23.287862,133.808728
min,4544.200195,0.335500,589.795776
25%,6304.800049,26.062875,625.345520
50%,10324.100098,28.454880,756.482239
75%,17465.538086,44.477501,854.780701
max,26328.550781,146.529999,1000.000000


Forward-filled cells: 0
Per-asset fill counts: {'equity': 0, 'gold': 0, 'bond': 0}
Dates dropped: 0
Data file SHA-256: bab5a877c16b189640504d150d2c37b805c8f17a2426e588c9a71b03e15391b7


### 4. Data Validation

The `validate_price_data()` function enforces all 8 contract rules.

In [3]:
from regime_shift.validation import validate_price_data

validated = validate_price_data(
    prices,
    required_cols=config.core_assets,
    allow_missing=False,
    forward_fill_limit=3,
    check_bfill=True,
)
print(f"Validation passed: {len(validated)} rows, {len(validated.columns)} columns")
print(f"Asset columns: {list(validated.columns)}")


Column 'bond' has 57% values equal to their successor - possible backward-fill detected. Verify raw data source.


check_forward_fill_limit called without a fill mask - falling back to identical-value heuristic (unreliable for low-volatility assets like bonds). Pass fill_mask from the data pipeline for accurate results.


Validation passed: 4090 rows, 3 columns
Asset columns: ['equity', 'gold', 'bond']


### 5. Leakage-Safe Features

Seven base features (plus 2 optional VIX features) computed with strictly trailing windows. Scaler is fit only on the training window.

In [4]:
from regime_shift.features import compute_raw_features, drop_feature_warmup
from regime_shift.features import fit_feature_scaler, transform_features

feature_cfg = config.feature_config
raw_features = compute_raw_features(prices, config=feature_cfg)
raw_features = drop_feature_warmup(
    raw_features, minimum_observations=feature_cfg.minimum_feature_observations,
    config=feature_cfg,
)
print(f"Features after warmup: {len(raw_features)} rows, {len(raw_features.columns)} columns")
print(f"Feature columns: {list(raw_features.columns)}")
display(raw_features.head())


Features after warmup: 3858 rows, 7 columns
Feature columns: ['equity_log_return_1d', 'equity_momentum_21d', 'equity_momentum_63d', 'equity_volatility_21d', 'equity_volatility_ratio_21_63', 'equity_gold_correlation_63d', 'equity_bond_correlation_63d']


,equity_log_return_1d,equity_momentum_21d,equity_momentum_63d,equity_volatility_21d,equity_volatility_ratio_21_63,equity_gold_correlation_63d,equity_bond_correlation_63d
Date,,,,,,,
2010-04-08,-0.013147,0.035217,0.013809,0.109581,0.682109,0.370115,0.045790
2010-04-09,0.010744,0.051014,0.015887,0.111588,0.692486,0.366416,0.045650
2010-04-12,-0.004121,0.043675,0.010962,0.113797,0.705176,0.362208,0.045584
2010-04-13,-0.003142,0.036925,0.011372,0.115067,0.713209,0.363533,0.055894
2010-04-15,-0.009314,0.026591,0.005501,0.121256,0.747142,0.358337,0.047337


### 6. Walk-Forward Gaussian HMM

A 3-state `GaussianHMM` with diagonal covariance is fit on each 252-day rolling training window. Only data through `t-1` is used.

In [5]:
from regime_shift.features import fit_feature_scaler, transform_features
from regime_shift.regime_model import fit_hmm

train_end = raw_features.index[min(config.train_window, len(raw_features) - 1)]
train_features = raw_features.loc[:train_end].tail(config.train_window)

scaler = fit_feature_scaler(train_features)
scaled_train = transform_features(scaler, train_features)

hmm, hidden_states, trans_mat = fit_hmm(
    scaled_train, train_features, config=config.hmm_config,
)
print(f"HMM converged: {hmm.monitor_.converged}")
print(f"Iterations: {hmm.monitor_.n_iter}")
print(f"Log-likelihood: {hmm.score(scaled_train.values):.2f}")
print(f"Transition matrix:")
display(trans_mat)


HMM converged: True
Iterations: 200
Log-likelihood: -2183.50
Transition matrix:


,Bull,Bear,Crisis
Bull,0.965793,0.003121,0.031086
Bear,0.000015,0.006469,0.993517
Crisis,0.035342,0.903397,0.061262


### 7. Bull/Bear/Crisis Interpretation

Numeric HMM states are mapped to interpretable labels using training-period volatility, momentum, and VIX statistics.

In [6]:
from regime_shift.regime_model import predict_current_state

solution = predict_current_state(
    hmm, scaler, train_features, train_features,
    config=config.hmm_config,
)

print(f"Current regime: {solution.regime}")
print(f"Regime probabilities: {dict(solution.probabilities)}")
print(f"Convergence: {solution.convergence}")
print(f"Iterations: {solution.n_iter}")
print(f"Transition matrix:")
display(solution.transition_matrix)


Current regime: Bull
Regime probabilities: {'Bull': np.float64(0.9995057336615757), 'Bear': np.float64(2.731901859600729e-05), 'Crisis': np.float64(0.00046694731960755044)}
Convergence: True
Iterations: 200
Transition matrix:


,Bull,Bear,Crisis
Bull,0.965793,0.003121,0.031086
Bear,0.000015,0.006469,0.993517
Crisis,0.035342,0.903397,0.061262


### 8. CVXPY Regime-Conditioned Portfolio Optimization

The optimiser selects one of three regime-specific convex objectives and applies regime-specific constraints.

In [7]:
from regime_shift.portfolio import optimize_portfolio

asset_returns = prices[["equity", "gold", "bond"]].pct_change().iloc[1:]
est_returns = asset_returns.loc[:train_end].tail(
    config.portfolio_config.estimation_lookback
)

port_sol = optimize_portfolio(
    regime=solution.regime,
    returns_through_date=est_returns,
    previous_weights=None,
    config=config.portfolio_config,
)

print(f"Regime: {port_sol.regime}")
print(f"Solver: {port_sol.solver}")
print(f"Status: {port_sol.status}")
print(f"Weights: equity={port_sol.weights['equity']:.4f}, gold={port_sol.weights['gold']:.4f}, bond={port_sol.weights['bond']:.4f}")


Regime: Bull
Solver: CLARABEL
Status: optimal
Weights: equity=0.6500, gold=0.3500, bond=0.0000


### 9. Transaction Costs

**Initial allocation:** turnover = sum |w_i| (full L1).

**Subsequent rebalance:** turnover = 0.5 * sum |w_i - w_{i-1}^{unadj}| (half-L1).

**Net return:** r_net = (1 - c) * (1 + r_gross) - 1 where c = turnover * bps/10000.

No cost on non-rebalance dates.

In [8]:
config.transaction_cost_bps = 5.0
cost_rate = config.transaction_cost_bps / 10000.0
print(f"Transaction cost rate: {cost_rate:.6f} ({config.transaction_cost_bps} bps)")


Transaction cost rate: 0.000500 (5.0 bps)


### 10. Benchmarks: 60/40 and Equal Weight

Two static benchmarks run with the same rebalance dates and cost convention as the strategy.

In [9]:
from regime_shift.benchmarks import static_60_40_weights, equal_weight_weights

static_6040 = static_60_40_weights()
equal_wt = equal_weight_weights()
print(f"Static 60/40: {static_6040}")
print(f"Equal Weight:  {equal_wt}")


Static 60/40: {'equity': 0.6, 'gold': 0.0, 'bond': 0.4}
Equal Weight:  {'equity': 0.3333333333333333, 'gold': 0.3333333333333333, 'bond': 0.3333333333333333}


### 11. Walk-Forward Timing — Full Backtest

The `run_walk_forward_backtest()` function executes a single chronological loop. At each rebalance date it fits the scaler, HMM, and optimiser using only data through `t-1`.

In [10]:
from regime_shift.backtest import run_walk_forward_backtest, run_benchmark
config.minimum_training_observations = 126
config.transaction_cost_bps = 5.0

strategy = run_walk_forward_backtest(
    prices=prices, config=config,
    transaction_cost_bps=5.0,
    risk_free_rate=0.0,
)

print(f"Backtest complete.")
print(f"Trading days: {len(strategy.net_returns)}")
print(f"Date range: {strategy.net_returns.index[0].date()} to {strategy.net_returns.index[-1].date()}")
print(f"Successful rebalances: {strategy.rebalance_flags.sum()}")

regime_counts = strategy.regime_series.value_counts()
for regime in ["Bull", "Bear", "Crisis"]:
    print(f"  {regime}: {regime_counts.get(regime, 0)} days")


Skipping rebalance on 2010-05-10 00:00:00: only 21 training observations, need 126.


Skipping rebalance on 2010-06-08 00:00:00: only 42 training observations, need 126.


Skipping rebalance on 2010-07-07 00:00:00: only 63 training observations, need 126.


Skipping rebalance on 2010-08-05 00:00:00: only 84 training observations, need 126.


Skipping rebalance on 2010-09-03 00:00:00: only 105 training observations, need 126.


...

Model is not converging.  Current: -1500.2978350913295 is not greater than -1500.2978263642024. Delta is -8.727127124075196e-06


Model is not converging.  Current: -1461.92506913205 is not greater than -1461.924491035387. Delta is -0.0005780966630481998


Model is not converging.  Current: -1461.7496300137366 is not greater than -1461.7496056226883. Delta is -2.4391048327743192e-05


Model is not converging.  Current: -1759.5500222161406 is not greater than -1759.5496217155217. Delta is -0.0004005006189800042


Backtest complete.
Trading days: 3732
Date range: 2010-10-05 to 2026-07-27
Successful rebalances: 178
  Bull: 1464 days
  Bear: 1218 days
  Crisis: 1050 days


In [11]:
bench_6040 = run_benchmark(
    prices=prices, weights=static_60_40_weights(),
    config=config,
    transaction_cost_bps=5.0,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
bench_ew = run_benchmark(
    prices=prices, weights=equal_weight_weights(),
    config=config,
    transaction_cost_bps=5.0,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
benchmarks = {"Static 60/40": bench_6040, "Equal Weight": bench_ew}
print("Benchmarks complete.")


Benchmarks complete.


### 12. Performance Metrics

Six-row performance summary computed from real market data.

In [12]:
from regime_shift.metrics import compute_performance_metrics
import pandas as pd

def _metrics_for(label, returns, gross_returns, turnover, costs, rf=0.0):
    m = compute_performance_metrics(
        returns, gross_returns=gross_returns,
        turnover=turnover, transaction_costs=costs,
        risk_free_rate=rf,
    )
    d = m.to_dict()
    d["Strategy"] = label
    return d

rows = [
    _metrics_for("RegimeShift Gross", strategy.gross_returns,
                 strategy.gross_returns, strategy.turnover,
                 strategy.transaction_costs),
    _metrics_for("RegimeShift Net", strategy.net_returns,
                 strategy.gross_returns, strategy.turnover,
                 strategy.transaction_costs),
    _metrics_for("Static 60/40 Gross", bench_6040.gross_returns,
                 bench_6040.gross_returns, bench_6040.turnover,
                 bench_6040.transaction_costs),
    _metrics_for("Static 60/40 Net", bench_6040.net_returns,
                 bench_6040.gross_returns, bench_6040.turnover,
                 bench_6040.transaction_costs),
    _metrics_for("Equal Weight Gross", bench_ew.gross_returns,
                 bench_ew.gross_returns, bench_ew.turnover,
                 bench_ew.transaction_costs),
    _metrics_for("Equal Weight Net", bench_ew.net_returns,
                 bench_ew.gross_returns, bench_ew.turnover,
                 bench_ew.transaction_costs),
]

perf_df = pd.DataFrame(rows)
perf_df = perf_df[[
    "Strategy", "Total Return", "CAGR", "Annualised Volatility",
    "Sharpe", "Sortino", "Maximum Drawdown", "Calmar",
    "Total Turnover", "Annualised Turnover", "Transaction Cost Drag",
]]
display(perf_df)

import os
os.makedirs("results", exist_ok=True)
perf_df.to_csv("results/performance_summary.csv", index=False)


,Strategy,Total Return,CAGR,Annualised Volatility,Sharpe,Sortino,Maximum Drawdown,Calmar,Total Turnover,Annualised Turnover,Transaction Cost Drag
0,RegimeShift Gross,1.8635,0.0736,0.1861,0.4689,0.5373,0.3601,0.2044,56.2845,3.8006,0.0000
1,RegimeShift Net,1.7840,0.0716,0.1861,0.4587,0.5260,0.3606,0.1985,56.2845,3.8006,0.0795
2,Static 60/40 Gross,1.9405,0.0722,0.0977,0.7621,0.7342,0.2330,0.3097,2.5740,0.1663,0.0000
3,Static 60/40 Net,1.9367,0.0721,0.0977,0.7613,0.7333,0.2330,0.3093,2.5740,0.1663,0.0038
4,Equal Weight Gross,2.7874,0.0898,0.1679,0.5912,0.6786,0.3288,0.2732,3.0619,0.1978,0.0000
5,Equal Weight Net,2.7816,0.0897,0.1679,0.5906,0.6779,0.3288,0.2729,3.0619,0.1978,0.0058


### 13. Charts and Robustness Checks

Six charts are generated by `generate_all_charts()`.

In [13]:
from regime_shift.plots import generate_all_charts
import os
os.makedirs("results", exist_ok=True)

saved = generate_all_charts(
    result=strategy,
    benchmarks=benchmarks,
    prices=prices,
    output_dir="results",
)
print(f"Saved {len(saved)} charts:")
for p in saved:
    print(f"  {p}")


Saved 6 charts:
  results\regime_price_chart.png
  results\transition_matrix.png
  results\equity_curves.png
  results\drawdowns.png
  results\portfolio_weights.png
  results\regime_probabilities.png


### 14. Conclusions and Limitations

**Strengths:**
- Fully leakage-safe walk-forward pipeline
- Deterministic regime mapping based on training-period statistics
- CVXPY regime-specific convex optimization
- Exact transaction-cost drag definition
- Automated test suite covering timing, metrics, charts, CLI, and notebook

**Limitations:**
- Bond proxy: LIQUIDBEES.NS — Nippon India Liquid Bees, INR-denominated liquid-bond / cash-equivalent proxy (NOT a sovereign bond or 10-year G-Sec; short duration, no long-term rate hedge). Short duration means no long-term rate hedge.
- Gaussian HMM assumes continuous Gaussian emissions — Student-t distributions may better capture fat tails.
- VIX is omitted; without it, Crisis detection relies solely on volatility and momentum.
- Three assets only; broader diversification requires expanding core_assets.
- No FX conversion — all assets are INR-denominated.
- The 5 bps results are the official submission figures; the 10 bps sensitivity is a robustness check, not a second submission.

**Sensitivity (5 bps vs 10 bps) — real data:**

| Cost | Sharpe | CAGR | Max DD | Cost Drag |
|---|---|---|---|---|
| 5 bps | 0.459 | 0.0716 | 0.3606 | 0.0795 |
| 10 bps | 0.448 | 0.0695 | 0.3612 | 0.1568 |

Doubling transaction costs from 5 to 10 bps had a limited effect on Sharpe and CAGR in this sample, although cumulative cost drag increased materially. This sensitivity result does not guarantee future robustness.

**Transition matrix sanity:** The matrix is a valid stochastic matrix (rows sum to 1, no identity placeholder).

**Drawdown math:** Drawdowns are computed from (1+r).cumprod() / cummax - 1 — never from r.cumprod().

**Daily drifted weights:** Portfolio-weight chart shows actual post-drift weights, not just target weights at rebalance dates.

**Regime probability order:** Probabilities are always in the explicit Bull / Bear / Crisis order — never sorted alphabetically.

**HMM inference consistency:** `predict_current_state` receives the same rolling training window used for fitting — never the full historical dataset.

**Forward-fill policy:** limit = 3 days. A naturally constant price series is not forward-filled; the 0 filled cells in this dataset were verified via an explicit fill mask.

In [14]:
# No additional code needed — sensitivity was run externally.
print("5 bps vs 10 bps — real data results shown in the table above.")
print(f"Forward-fill limit: 3 days")
print(f"Total forward-filled cells: 0")
print("Per-asset fill counts:", {'equity': 0, 'gold': 0, 'bond': 0})
print(f"Dates dropped (residual NaNs): 0")
print(f"HMM uses rolling training window only (not full dataset)")


5 bps vs 10 bps — real data results shown in the table above.
Forward-fill limit: 3 days
Total forward-filled cells: 0
Per-asset fill counts: {'equity': 0, 'gold': 0, 'bond': 0}
Dates dropped (residual NaNs): 0
HMM uses rolling training window only (not full dataset)
